[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/pypath/blob/main/notebooks/module5/01-ml-concepts.ipynb)

# Module 5 — Lesson 1: Machine Learning Concepts

**Module:** 5 — Machine Learning Foundations | **Time:** 20 minutes

## Learning Objectives

By the end of this lesson you will be able to:

- Distinguish between supervised, unsupervised, and reinforcement learning
- Identify regression, classification, and clustering problem types
- Split data correctly using `train_test_split` with stratification
- Explain the bias-variance tradeoff and its practical consequences
- Diagnose overfitting and underfitting using learning curves

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.datasets import load_iris, load_breast_cancer, make_blobs
from sklearn.model_selection import train_test_split, cross_val_score, validation_curve, learning_curve
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})
print('Libraries loaded successfully.')

## 1. Taxonomy of Machine Learning

Machine learning is broadly divided into three paradigms based on the type of feedback signal available during training.

| Paradigm | Signal | Typical Tasks | Examples |
|---|---|---|---|
| **Supervised** | Labelled (X, y) pairs | Regression, Classification | House prices, Spam detection |
| **Unsupervised** | Unlabelled X only | Clustering, Dimensionality reduction | Customer segments, Topic modelling |
| **Reinforcement** | Reward from environment | Control, Games | AlphaGo, Robotics |

Within **supervised learning**:
- **Regression** — target `y` is continuous (e.g., temperature, price)
- **Classification** — target `y` is a discrete class label (e.g., cat/dog, malignant/benign)

In [ ]:
# --- Taxonomy illustration ---
categories = {
    'Supervised\n(Regression)': ['Linear Regression', 'SVR', 'Random Forest\nRegressor'],
    'Supervised\n(Classification)': ['Logistic Regression', 'SVM', 'Neural Networks'],
    'Unsupervised\n(Clustering)': ['K-Means', 'DBSCAN', 'Hierarchical'],
    'Unsupervised\n(Dim. Reduction)': ['PCA', 'UMAP', 't-SNE'],
}

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
colors = ['#4C9BE8', '#4C9BE8', '#E87B4C', '#E87B4C']

for ax, (title, algorithms), color in zip(axes, categories.items(), colors):
    ax.set_xlim(0, 1)
    ax.set_ylim(0, len(algorithms) + 1)
    ax.set_facecolor(color + '22')
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.axis('off')
    for i, algo in enumerate(algorithms, 1):
        ax.text(0.5, i, algo, ha='center', va='center', fontsize=9,
                bbox=dict(boxstyle='round,pad=0.4', facecolor=color, alpha=0.6))

plt.suptitle('Machine Learning Taxonomy', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('Taxonomy chart rendered.')

## 2. Train / Test Split with Stratification

Before training any model you must hold out a portion of data for unbiased evaluation. Key considerations:

- **`test_size`** — typically 0.2 (20 %) for moderate datasets
- **`stratify=y`** — preserves class proportions in both splits (essential for imbalanced datasets)
- **`random_state`** — ensures reproducibility

Without stratification, a random split could accidentally put 90 % of one class in the training set.

In [ ]:
# Load breast cancer dataset (binary classification)
data = load_breast_cancer()
X, y = data.data, data.target

print(f'Full dataset shape: {X.shape}')
print(f'Class distribution (full): {np.bincount(y)} — {np.bincount(y)/len(y)*100}')

# Split WITHOUT stratification
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'\nWithout stratify — train class ratio: {np.bincount(y_tr)/len(y_tr)*100.round(1)}')
print(f'Without stratify — test  class ratio: {np.bincount(y_te)/len(y_te)*100.round(1)}')

# Split WITH stratification
X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'\nWith    stratify — train class ratio: {np.bincount(y_tr_s)/len(y_tr_s)*100.round(1)}')
print(f'With    stratify — test  class ratio: {np.bincount(y_te_s)/len(y_te_s)*100.round(1)}')

print('\nTrain size:', X_tr_s.shape[0], '| Test size:', X_te_s.shape[0])

## 3. Cross-Validation

A single train/test split is sensitive to which samples happen to land in each partition. **k-fold cross-validation** repeatedly trains and evaluates on k different partitions, giving a more reliable performance estimate.

`cross_val_score` returns k scores — report mean ± standard deviation.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000))
])

scores = cross_val_score(pipe, X, y, cv=5, scoring='accuracy')
print('5-Fold Cross-Validation Accuracy:')
for i, s in enumerate(scores, 1):
    print(f'  Fold {i}: {s:.4f}')
print(f'\nMean: {scores.mean():.4f}  |  Std: {scores.std():.4f}')
print(f'95% CI: [{scores.mean()-2*scores.std():.4f}, {scores.mean()+2*scores.std():.4f}]')

## 4. Bias-Variance Tradeoff

Every model's generalisation error decomposes into three terms:

```
Total Error = Bias² + Variance + Irreducible Noise
```

- **High Bias (Underfitting)** — model is too simple; misses the true pattern in both train and test data
- **High Variance (Overfitting)** — model is too complex; memorises training noise; fails on new data
- **Sweet spot** — the model complexity that minimises total error

Polynomial degree is a classic proxy for model complexity.

In [ ]:
# Generate a noisy sine-wave dataset
np.random.seed(0)
X_1d = np.sort(np.random.uniform(0, 1, 50))
y_1d = np.sin(2 * np.pi * X_1d) + np.random.normal(0, 0.3, 50)

X_plot = np.linspace(0, 1, 300).reshape(-1, 1)

degrees = [1, 4, 15]
titles  = ['Degree 1 — High Bias\n(Underfitting)', 'Degree 4 — Good Fit', 'Degree 15 — High Variance\n(Overfitting)']
colors  = ['#e74c3c', '#2ecc71', '#9b59b6']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, deg, title, color in zip(axes, degrees, titles, colors):
    pipe_poly = Pipeline([
        ('poly', PolynomialFeatures(degree=deg)),
        ('scaler', StandardScaler()),
        ('reg', LinearRegression())
    ])
    pipe_poly.fit(X_1d.reshape(-1, 1), y_1d)
    y_pred = pipe_poly.predict(X_plot)
    train_mse = mean_squared_error(y_1d, pipe_poly.predict(X_1d.reshape(-1, 1)))

    ax.scatter(X_1d, y_1d, s=25, alpha=0.6, color='steelblue', label='Data')
    ax.plot(X_plot, np.sin(2*np.pi*X_plot), 'k--', lw=1.5, label='True function')
    ax.plot(X_plot, y_pred, color=color, lw=2, label=f'Poly deg={deg}')
    ax.set_title(f'{title}\nTrain MSE={train_mse:.3f}', fontsize=10)
    ax.set_ylim(-2, 2)
    ax.legend(fontsize=8)
    ax.set_xlabel('x')

plt.suptitle('Bias-Variance Tradeoff via Polynomial Degree', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Validation Curve — Finding the Sweet Spot

`validation_curve` trains the model across a range of a single hyperparameter and records training and validation scores. The gap between the two curves reveals whether the model is underfitting or overfitting.

In [ ]:
from sklearn.model_selection import validation_curve

param_range = np.arange(1, 12)

pipe_vc = Pipeline([
    ('poly', PolynomialFeatures()),
    ('scaler', StandardScaler()),
    ('reg', Ridge(alpha=1e-3))
])

train_scores, val_scores = validation_curve(
    pipe_vc, X_1d.reshape(-1, 1), y_1d,
    param_name='poly__degree',
    param_range=param_range,
    cv=5,
    scoring='neg_mean_squared_error'
)

train_mean = -train_scores.mean(axis=1)
val_mean   = -val_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_std    = val_scores.std(axis=1)

best_deg = param_range[np.argmin(val_mean)]

plt.figure(figsize=(9, 5))
plt.plot(param_range, train_mean, 'o-', color='steelblue', label='Train MSE')
plt.plot(param_range, val_mean,   's-', color='tomato',    label='Validation MSE')
plt.fill_between(param_range, train_mean - train_std, train_mean + train_std, alpha=0.15, color='steelblue')
plt.fill_between(param_range, val_mean   - val_std,   val_mean   + val_std,   alpha=0.15, color='tomato')
plt.axvline(best_deg, linestyle='--', color='green', label=f'Best degree = {best_deg}')
plt.xlabel('Polynomial Degree')
plt.ylabel('MSE')
plt.title('Validation Curve — Polynomial Degree vs MSE')
plt.legend()
plt.tight_layout()
plt.show()
print(f'Best polynomial degree: {best_deg}  |  Validation MSE: {val_mean[best_deg-1]:.4f}')

## 6. Learning Curves — Data vs Performance

Learning curves plot model performance as the **training set size grows**. They answer: *"Will collecting more data help?"*

- **Underfitting** — both curves are high (bad performance); more data does not help much
- **Overfitting** — large gap between train and validation; more data gradually closes the gap
- **Good fit** — curves converge at a low error value

In [ ]:
from sklearn.model_selection import learning_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, deg, title in zip(axes, [1, 9], ['Degree 1 (Underfitting)', 'Degree 9 (Overfitting)']):
    pipe_lc = Pipeline([
        ('poly', PolynomialFeatures(degree=deg)),
        ('scaler', StandardScaler()),
        ('reg', Ridge(alpha=1e-6))
    ])
    train_sizes, train_sc, val_sc = learning_curve(
        pipe_lc, X_1d.reshape(-1, 1), y_1d,
        train_sizes=np.linspace(0.15, 1.0, 8),
        cv=5, scoring='neg_mean_squared_error'
    )
    tr_m = -train_sc.mean(axis=1)
    va_m = -val_sc.mean(axis=1)

    ax.plot(train_sizes, tr_m, 'o-', color='steelblue', label='Train')
    ax.plot(train_sizes, va_m, 's-', color='tomato', label='Validation')
    ax.fill_between(train_sizes, tr_m-train_sc.std(axis=1), tr_m+train_sc.std(axis=1), alpha=0.15, color='steelblue')
    ax.fill_between(train_sizes, va_m-val_sc.std(axis=1),   va_m+val_sc.std(axis=1),   alpha=0.15, color='tomato')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Training set size')
    ax.set_ylabel('MSE')
    ax.legend()

plt.suptitle('Learning Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Clustering — Unsupervised Example

In unsupervised learning there is no label `y`. K-Means groups data points into clusters by minimising within-cluster distances.

In [ ]:
from sklearn.cluster import KMeans

X_blobs, y_blobs = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)
kmeans = KMeans(n_clusters=4, random_state=42, n_init='auto')
labels = kmeans.fit_predict(X_blobs)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], s=20, alpha=0.7, color='steelblue')
axes[0].set_title('Raw Data — No Labels (Unsupervised)')

scatter = axes[1].scatter(X_blobs[:, 0], X_blobs[:, 1], c=labels, cmap='tab10', s=20, alpha=0.7)
axes[1].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
                s=200, marker='X', c='red', zorder=5, label='Centroids')
axes[1].set_title('K-Means Clustering Result (k=4)')
axes[1].legend()

for ax in axes:
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.tight_layout()
plt.show()
print('Inertia (within-cluster sum of squares):', round(kmeans.inertia_, 2))

## Practice Exercises

**Exercise 1 — Stratified Split Verification**
Load `load_iris()` from `sklearn.datasets`. Perform a stratified 70/30 train/test split. Print the class distribution (percentage) for both the training and test sets, and verify the distributions match.

**Exercise 2 — Validation Curve for a Classifier**
Using `load_breast_cancer()`, build a `Pipeline` with `StandardScaler` and `LogisticRegression`. Plot a validation curve over `logisticregression__C` values `[0.001, 0.01, 0.1, 1, 10, 100]`. Identify the C value that minimises overfitting.

**Exercise 3 — Bias-Variance Narrative**
For the polynomial regression example in section 4, compute and print the **test MSE** (using the held-out 20 % of the original 1-D dataset) for degrees 1, 4, and 15. Create a bar chart of Train MSE vs Test MSE for each degree and explain in a comment what the chart reveals about bias and variance.